# Drivetrain root-cause debug notebook

Works through the failure modes in order, cheapest test first. Each step says what **good** and **bad** look like and where to go next.

**Before starting:** firmware v3 flashed (`sketches/drivetrain/drivetrain.ino`), bot **on blocks** (wheels off the ground) unless a step says otherwise, and no other program holding the serial port (close Arduino IDE Serial Monitor).

| Step | Question it answers |
|---|---|
| 1 | Is the link clean and the right firmware running? |
| 2 | Do the encoders count only when wheels move, on the right channels? |
| 3 | Open-loop F/B/L/R: do both motors run symmetrically with no resets? |
| 4 | Does the forward→reverse transition (the historic killer) survive repeats? |
| 5 | Encoder-counted moves: does closed-loop sync work? |
| 6 | Open- vs closed-loop comparison → the verdict |

## Step 0 — Connect
Expect: `Connected. Firmware: drv8871-v3 built <today's date/time>`. A stale stamp means re-flash before continuing — nothing below is meaningful on old firmware.

In [ ]:
import time
import logging
import sys, os
# Repo root on the path regardless of where jupyter was launched from.
_here = os.getcwd()
sys.path.insert(0, os.path.dirname(_here) if os.path.basename(_here) == "tests" else _here)

PORT = "/dev/cu.usbserial-A5069RR4"

class LogCapture(logging.Handler):
    """Collects the bridge's WARNING+ messages so each step can show
    exactly what happened during ITS window."""
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.lines = []
    def emit(self, record):
        self.lines.append(record.getMessage())
    def take(self):
        out, self.lines = self.lines, []
        return out

logging.basicConfig(level=logging.WARNING, format="%(message)s")
capture = LogCapture()
logging.getLogger().addHandler(capture)

enc = {"l": 0, "r": 0}
def on_enc(l, r):
    enc["l"], enc["r"] = l, r

from drivetrain import ArduinoBridge
bridge = ArduinoBridge(port=PORT, on_encoder=on_enc)
capture.take()  # discard connect-time boot message
print("Connected. Firmware:", bridge.firmware_build)

Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20


Connected. Firmware: drv8871-v3 built Aug  9 2026 08:40:20


### 🛑 Emergency stop / reconnect
Run the first cell any time to stop the motors. If the board hung (commands time out), run the second — closing and reopening the port pulls the RESET line and revives a hung chip.

In [9]:
bridge.stop()

Serial read failed — reader stopping


In [6]:
# Reconnect (also revives a hung board via the port-open reset)
try:
    bridge.close()
except Exception:
    pass
bridge = ArduinoBridge(port=PORT, on_encoder=on_enc)
capture.take()
print("Reconnected. Firmware:", bridge.firmware_build)

stop(): no ACK received (already disconnected?)
Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20


Reconnected. Firmware: drv8871-v3 built Aug  9 2026 08:40:20


## Step 1 — Link health at idle (10 s, motors off)
**Good:** counts frozen, 0 corrupted frames, no warnings.
**Bad:** counts creeping = encoder lines picking up noise at idle (wiring). Corrupted frames at idle = serial/USB noise even without motors — fix USB path first, everything else will be muddied by it.

In [ ]:
start = (enc["l"], enc["r"])
bad0, warn0 = bridge._parser.bad_frames, bridge.warnings_seen
time.sleep(10)
print(f"count creep : ΔL={enc['l']-start[0]:+d} ΔR={enc['r']-start[1]:+d}   (want 0, 0)")
print(f"bad frames  : {bridge._parser.bad_frames - bad0}   (want 0)")
print(f"warnings    : {bridge.warnings_seen - warn0}   (want 0)")
for m in capture.take():
    print("  !", m)

count creep : ΔL=+0 ΔR=+0   (want 0, 0)
bad frames  : 0   (want 0)
warnings    : 0   (want 0)


Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


## Step 2 — Hand-spin encoder mapping (30 s, motors off)
Spin the **left** wheel by hand, then the **right**, then wiggle each encoder cable.
**Good:** left wheel moves only the L column, right only R, smooth counts, nothing when wiggling.
**Also check (quadrature, fw v4+):** forward roll must count UP on both sides — a side counting down needs its `ENC_x_INVERT` flipped in the firmware.
**Bad:** swapped columns = crossed encoder plugs (fix before any sync test — the controller punishes the wrong wheel). Bursts while wiggling = loose connector on that channel.

In [9]:
print("Spin wheels by hand now (30 s)...")
print(f"{'t':>4} {'L':>8} {'R':>8}")
t_end = time.monotonic() + 30
while time.monotonic() < t_end:
    print(f"{30 - (t_end - time.monotonic()):4.0f} {enc['l']:8d} {enc['r']:8d}", end="\r")
    time.sleep(0.25)
print()
for m in capture.take():
    print("  !", m)

Spin wheels by hand now (30 s)...
   t        L        R
  30        0        0
  ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
  ! Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


## Step 3 — Open-loop singles: F, B, L, R
Each command drives 3 s at PWM 127, soft-stops, and reports. Runs are separated so you can watch each one.
**Good:** |rate L| ≈ |rate R| within ~15 % on every direction, zero resets/hangs.
**Bad:** one wheel consistently slower off the ground = mechanical drag or weak motor on that side (swap motor leads L↔R at the drivers to see if it follows the motor). A reset/hang logged here = electrical, note WHICH direction triggered it.

In [7]:
open_loop = {}

def run_intent(direction, seconds=3.0, speed_pwm=127):
    capture.take()
    resets0 = bridge.resets_seen
    before = (enc["l"], enc["r"])
    method = {"F": bridge.forward, "B": bridge.backward,
              "L": bridge.left, "R": bridge.right}[direction]
    ok = True
    try:
        method()
        time.sleep(seconds)
    except Exception as exc:
        ok = False
        print(f"  COMMAND FAILED: {exc}")
    finally:
        try:
            bridge.stop()
        except Exception as exc:
            ok = False
            print(f"  STOP FAILED (board hung?): {exc}")
    time.sleep(0.4)
    dl, dr = enc["l"] - before[0], enc["r"] - before[1]
    logs = capture.take()
    print(f"{direction}: ΔL={dl:+6d} ΔR={dr:+6d}   "
          f"rate L={abs(dl)/seconds:5.0f}/s R={abs(dr)/seconds:5.0f}/s   "
          f"resets={bridge.resets_seen - resets0}")
    for m in logs:
        print("   !", m)
    open_loop[direction] = (dl, dr, ok)
    return ok

bridge.set_speed_pwm(127)
for d in ("F", "B", "L", "R"):
    run_intent(d)
    time.sleep(1.5)

F: ΔL= +1667 ΔR= +1490   rate L=  556/s R=  497/s   resets=0


Arduino has dropped 1 corrupted frame(s) — electrical noise on the serial line (commands unaffected)
Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


B: ΔL= -1583 ΔR= -1390   rate L=  528/s R=  463/s   resets=1
   ! Arduino has dropped 1 corrupted frame(s) — electrical noise on the serial line (commands unaffected)
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
stop(): no ACK received (already disconnected?)


L: ΔL=   -84 ΔR=   -15   rate L=   28/s R=    5/s   resets=1
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! stop(): no ACK received (already disconnected?)


Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)
Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
stop(): no ACK received (already disconnected?)


R: ΔL=    +0 ΔR=   -85   rate L=    0/s R=   28/s   resets=2
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! stop(): no ACK received (already disconnected?)


In [ ]:
open_loop = {}

def run_intent(direction, seconds=3.0, speed_pwm=127):
    capture.take()
    resets0 = bridge.resets_seen
    before = (enc["l"], enc["r"])
    method = {"F": bridge.forward, "B": bridge.backward,
              "L": bridge.left, "R": bridge.right}[direction]
    ok = True
    try:
        method()
        time.sleep(seconds)
    except Exception as exc:
        ok = False
        print(f"  COMMAND FAILED: {exc}")
    finally:
        try:
            bridge.stop()
        except Exception as exc:
            ok = False
            print(f"  STOP FAILED (board hung?): {exc}")
    time.sleep(0.4)
    dl, dr = enc["l"] - before[0], enc["r"] - before[1]
    logs = capture.take()
    print(f"{direction}: ΔL={dl:+6d} ΔR={dr:+6d}   "
          f"rate L={abs(dl)/seconds:5.0f}/s R={abs(dr)/seconds:5.0f}/s   "
          f"resets={bridge.resets_seen - resets0}")
    for m in logs:
        print("   !", m)
    open_loop[direction] = (dl, dr, ok)
    return ok

bridge.set_speed_pwm(127)
for d in ("F", "B", "L", "R"):
    run_intent(d)
    time.sleep(1.5)

Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


F: ΔL=   +90 ΔR=  +104   rate L=   30/s R=   35/s   resets=1
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


B: ΔL=    +4 ΔR=   +22   rate L=    1/s R=    7/s   resets=1
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


L: ΔL=    -3 ΔR=   -10   rate L=    1/s R=    3/s   resets=1
   ! Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
   ! Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)
  COMMAND FAILED: No ACK from Arduino within 1.4 s and no boot frame afterwards — the board is likely HUNG (a supply dip with the brown-out detector disabled locks the chip up instead of resetting it). Reopen the port or power-cycle.


KeyboardInterrupt: 

Arduino firmware: drv8871-v3 built Aug  9 2026 08:40:20
Arduino reset while idle — power-on (the Arduino's 5V supply dropped completely — USB power was interrupted: check the USB cable, connector, and port); brown-out (the Arduino's 5V rail sagged below ~2.7V — something is dragging down or coupling into the 5V supply)


## Step 4 — Stress the killer transition: forward → reverse ×5
This is the exact moment that has been hanging the board (brake + direction flip). Five back-to-back cycles.
**Good:** 5/5 survived, zero resets.
**Bad:** any hang/reset here with wheels OFF the ground is decisive — the transient alone (no load) disturbs the logic supply. Do the 5 V min/max meter measurement during this cell, then: motor terminal caps, encoder-harness rerouting, BOD fuse.
Repeat this cell **on the ground** afterwards — surviving off-ground but dying on-ground points at load current (buck sag / ILIM) instead.

In [ ]:
survived = 0
resets0 = bridge.resets_seen
for i in range(1, 6):
    print(f"cycle {i}: ", end="")
    ok_f = run_intent("F", seconds=2.0)
    ok_b = run_intent("B", seconds=2.0) if ok_f else False
    if ok_f and ok_b:
        survived += 1
    else:
        print("  -> stopping stress test; reconnect (cell above) before continuing")
        break
    time.sleep(1.0)
print(f"\nSurvived {survived}/5 cycles, resets during test: {bridge.resets_seen - resets0}")

## Step 5 — Encoder-counted (closed-loop) moves
Same four directions, but now the firmware stops on encoder targets and runs the sync trim. ~1000 ticks ≈ 0.4 m of wheel travel.
**Good:** every move completes, |ΔL| ≈ |ΔR| ≈ target within ~10 %, elapsed ≈ open-loop time for the same distance.
**Bad, decoded:**
- `TIMEOUT` with one Δ far below target → that wheel (or its encoder) isn't keeping up — cross-check with Step 3's rates.
- One Δ way OVER target → that wheel waited for the other; the sync trim hit its authority limit. Compare with Step 6.
- `NOISE` → encoder channel corrupting under motor PWM: wiring/filtering.
- Hang → same electrical story as Step 4.

In [ ]:
closed_loop = {}
TICKS = 1000

def run_move(direction, ticks=TICKS, speed_pwm=127):
    capture.take()
    before = (enc["l"], enc["r"])
    t0 = time.monotonic()
    status = "ok"
    try:
        bridge.move(direction, speed_pwm, ticks)
    except Exception as exc:
        status = f"FAILED: {exc}"
    elapsed = time.monotonic() - t0
    time.sleep(0.4)
    dl, dr = enc["l"] - before[0], enc["r"] - before[1]
    print(f"{direction}: {status}")
    print(f"   elapsed={elapsed:5.2f}s  ΔL={dl:+6d} ΔR={dr:+6d}  target={ticks}")
    for m in capture.take():
        print("   !", m)
    closed_loop[direction] = (dl, dr, elapsed, status)

for d in ("F", "B", "L", "R"):
    run_move(d)
    time.sleep(1.5)

## Step 6 — Verdict: open-loop vs closed-loop
The comparison that separates the remaining suspects.

In [ ]:
print(f"{'dir':>4} {'open ΔL':>9} {'open ΔR':>9} {'ratio':>6}   {'closed ΔL':>9} {'closed ΔR':>9} {'ratio':>6}")
for d in ("F", "B", "L", "R"):
    o = open_loop.get(d)
    c = closed_loop.get(d)
    if not o or not c:
        continue
    orat = abs(o[0]) / max(abs(o[1]), 1)
    crat = abs(c[0]) / max(abs(c[1]), 1)
    print(f"{d:>4} {o[0]:9d} {o[1]:9d} {orat:6.2f}   {c[0]:9d} {c[1]:9d} {crat:6.2f}")

print("""
How to read this:
  open ratio ~1, closed ratio ~1      -> drivetrain + control healthy. Done.
  open ratio ~1, closed ratio BAD     -> motors match at equal PWM but the
                                         sync loop diverges: encoder data is
                                         lying under closed-loop conditions
                                         (dropout/noise on one channel) OR one
                                         driver current-limits when boosted.
  open ratio BAD everywhere           -> real mechanical/electrical asymmetry:
                                         swap the two motors' leads at the
                                         drivers; if the slow side follows the
                                         motor it's the motor/gearbox, if it
                                         stays it's the driver or its supply.
  anything hung/reset                 -> electrical transient story (Step 4):
                                         5V meter test, motor caps, BOD fuse.""")

## Cleanup

In [ ]:
bridge.close()
print("Port closed.")